In [ ]:
# Cell 1: Mount Google Drive first, install dependencies, and create project folders
ENV_NAME='v'
!pip -q install -U 'ultralytics>=8.3.0,<9' 'transformers>=4.45,<5' accelerate opencv-contrib-python-headless pandas scipy matplotlib tqdm lxml
import os,sys,json,math,csv,shutil,subprocess,platform,warnings
from pathlib import Path
import numpy as np,pandas as pd,cv2,torch
from tqdm.auto import tqdm
from time import perf_counter
from IPython.display import display,HTML
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT=Path('/content/drive/MyDrive/lane-opendrive-v3')
PROJECT_DIRS={name:PROJECT_ROOT/name for name in ['input','outputs','frames','visualizations','reports','opendrive','logs']}
for path in PROJECT_DIRS.values(): path.mkdir(parents=True,exist_ok=True)
ROOT=PROJECT_DIRS['outputs']; VIS_ROOT=ROOT/'visualizations'; VIS_ROOT.mkdir(parents=True,exist_ok=True); ARCHIVE_PATH=PROJECT_ROOT/'complete_results.zip'

DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
HALF=DEVICE=='cuda'
MAX_FRAMES=int(os.environ.get('MAX_FRAMES','0'))
FRAME_STRIDE=int(os.environ.get('FRAME_STRIDE','1'))
METRIC_SCALE_ENV=os.environ.get('METRIC_SCALE_M_PER_RELATIVE_UNIT')
METRIC_SCALE=float(METRIC_SCALE_ENV) if METRIC_SCALE_ENV else None

print(f'Drive mounted.')
print(f'Project root: {PROJECT_ROOT}')
print(f'Input folder: {PROJECT_DIRS["input"]}')
print('Manual workflow: copy an MP4 into the input folder, then run Cell 2.')
print(json.dumps({'device':DEVICE,'torch':torch.__version__,'cuda':torch.version.cuda,'metric_scale_status':'ASSUMED' if METRIC_SCALE is not None else 'UNKNOWN'},indent=2))

MODEL_SELECTION=[
 {'stage':'depth','selected':'Depth Anything V2 Small (Hugging Face)','repository':'https://huggingface.co/depth-anything/Depth-Anything-V2-Small-hf','checkpoint':'depth-anything/Depth-Anything-V2-Small-hf','input':'RGB frame','output':'relative inverse-depth','python':'3.9+','pytorch':'2.x','cuda':'optional','compile':'none','t4':'yes'},
 {'stage':'vehicles','selected':'Ultralytics YOLO11m','repository':'https://github.com/ultralytics/ultralytics','checkpoint':'yolo11m.pt','input':'RGB frame','output':'2D COCO detections','python':'3.8+','pytorch':'2.x','cuda':'optional','compile':'none','t4':'yes'},
 {'stage':'tracking','selected':'temporal IoU tracker with missed-frame prediction','repository':'https://github.com/ultralytics/ultralytics','checkpoint':'none','input':'detections','output':'persistent tracks','python':'3.8+','pytorch':'none','cuda':'none','compile':'none','t4':'yes'},
 {'stage':'lane','selected':'multi-lane geometric reconstruction from segmentation/marking evidence + depth + temporal tracking','repository':'https://huggingface.co/nvidia/segformer-b2-finetuned-cityscapes-1024-1024','checkpoint':'nvidia/segformer-b2-finetuned-cityscapes-1024-1024','input':'RGB frame','output':'observed/estimated/predicted lane boundaries','python':'3.9+','pytorch':'2.x','cuda':'optional','compile':'none','t4':'yes'},
 {'stage':'vo','selected':'OpenCV optical flow + essential matrix RANSAC','repository':'https://opencv.org','checkpoint':'none','input':'successive frames','output':'relative pose and speed','python':'3.8+','pytorch':'none','cuda':'none','compile':'none','t4':'yes'},
 {'stage':'opendrive','selected':'Evidence-derived OpenDRIVE writer and validator','repository':'https://www.asam.net/standards/detail/opendrive/','checkpoint':'none','input':'reconstructed geometry','output':'road.xodr','python':'3.8+','pytorch':'none','cuda':'none','compile':'none','t4':'yes'}]
print(json.dumps({'models':MODEL_SELECTION},indent=2))

In [ ]:
# Cell 2: manually placed Drive MP4 discovery, validation, and model initialization
from transformers import AutoImageProcessor,AutoModelForDepthEstimation,SegformerImageProcessor,SegformerForSemanticSegmentation
from ultralytics import YOLO

# After Cell 1, manually copy an MP4 into:
# /content/drive/MyDrive/lane-opendrive-v3/input/
input_candidates=sorted(PROJECT_DIRS['input'].glob('*.mp4'),key=lambda path:path.stat().st_mtime,reverse=True)
if not input_candidates: raise FileNotFoundError(f'No MP4 found in {PROJECT_DIRS["input"]}. Copy a front-facing MP4 there and rerun Cell 2.')
INPUT_VIDEO=str(input_candidates[0]); input_path=Path(INPUT_VIDEO)
cap=cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened(): raise RuntimeError(f'Selected MP4 could not be opened: {INPUT_VIDEO}')
FPS=float(cap.get(cv2.CAP_PROP_FPS) or 0); WIDTH=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); HEIGHT=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)); TOTAL=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
if FPS<=0 or WIDTH<=0 or HEIGHT<=0 or TOTAL<=0: raise RuntimeError(f'Selected MP4 has invalid metadata: {INPUT_VIDEO}')
print(json.dumps({'selected_input':INPUT_VIDEO,'fps':FPS,'resolution':(WIDTH,HEIGHT),'frames':TOTAL,'duration_s':TOTAL/FPS,'device':DEVICE},indent=2))
display(HTML('<b>Pipeline:</b> Input PASS &rarr; Calibration pending &rarr; Road/Lanes pending &rarr; Depth pending &rarr; Vehicles pending &rarr; Ego Motion pending &rarr; Topology pending &rarr; BEV/3D pending &rarr; OpenDRIVE pending &rarr; Validation pending'))
DEPTH_PROCESSOR=AutoImageProcessor.from_pretrained('depth-anything/Depth-Anything-V2-Small-hf'); DEPTH_MODEL=AutoModelForDepthEstimation.from_pretrained('depth-anything/Depth-Anything-V2-Small-hf').to(DEVICE).eval()
ROAD_PROCESSOR=SegformerImageProcessor.from_pretrained('nvidia/segformer-b2-finetuned-cityscapes-1024-1024'); ROAD_MODEL=SegformerForSemanticSegmentation.from_pretrained('nvidia/segformer-b2-finetuned-cityscapes-1024-1024').to(DEVICE).eval()
VEHICLE_MODEL=YOLO('yolo11m.pt'); COCO_VEHICLES={'car','truck','bus','motorcycle','bicycle','person'}
print('Models initialized successfully.')

In [ ]:
# Cell 3: depth, road segmentation, vanishing-point calibration, and 3D projection helpers
from PIL import Image
from scipy.signal import savgol_filter

def infer_depth(frame):
    rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
    inputs=DEPTH_PROCESSOR(images=Image.fromarray(rgb),return_tensors='pt').to(DEVICE)
    with torch.inference_mode(): pred=DEPTH_MODEL(**inputs).predicted_depth
    depth=torch.nn.functional.interpolate(pred.unsqueeze(1),size=frame.shape[:2],mode='bicubic',align_corners=False).squeeze().float().cpu().numpy()
    lo,hi=np.nanpercentile(depth,[1,99]); depth=np.clip((depth-lo)/(hi-lo+1e-6),0,1)
    return depth,{'status':'MEASURED','metric':False,'source':'Depth Anything V2 Small relative inverse-depth','unit':'relative_inverse_depth'}

def infer_road_mask(frame):
    rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB); inp=ROAD_PROCESSOR(images=Image.fromarray(rgb),return_tensors='pt').to(DEVICE)
    with torch.inference_mode(): logits=ROAD_MODEL(**inp).logits
    logits=torch.nn.functional.interpolate(logits,size=frame.shape[:2],mode='bilinear',align_corners=False)[0]
    labels={int(k):str(v).lower() for k,v in getattr(ROAD_MODEL.config,'id2label',{}).items()}; road_ids=[k for k,v in labels.items() if v in {'road','road surface'} or 'road' in v]
    if not road_ids: return np.zeros(frame.shape[:2],np.uint8),{'status':'FAIL','source':'SegFormer id2label has no road class'}
    pred=logits.argmax(0).cpu().numpy(); return np.isin(pred,road_ids).astype(np.uint8),{'status':'MEASURED','source':'SegFormer Cityscapes road class'}

def _vp(lines):
    if len(lines)<2: return None
    hs=[np.cross([x1,y1,1.],[x2,y2,1.]) for x1,y1,x2,y2 in lines]; _,_,vh=np.linalg.svd(np.asarray(hs)); p=vh[-1]
    return None if abs(p[2])<1e-8 else [float(p[0]/p[2]),float(p[1]/p[2])]

def estimate_calibration(frame,road_mask):
    h,w=frame.shape[:2]; cx,cy=w/2.,h/2.; edges=cv2.Canny(frame,70,160); raw=cv2.HoughLinesP(edges,1,np.pi/180,threshold=max(35,w//10),minLineLength=max(35,w//14),maxLineGap=25)
    road_lines=[]; vertical_lines=[]
    if raw is not None:
        # OpenCV wheels return either (N, 1, 4) or (N, 4), so normalize both.
        for x1,y1,x2,y2 in np.asarray(raw).reshape(-1,4):
            dx,dy=float(x2-x1),float(y2-y1)
            if abs(dx)<4 or max(y1,y2)<.45*h: continue
            slope=dy/(dx+1e-6)
            if abs(slope)>.25: road_lines.append([x1,y1,x2,y2])
            if abs(dx)<.45*abs(dy): vertical_lines.append([x1,y1,x2,y2])
    road_vp=_vp(road_lines); vertical_vp=_vp(vertical_lines); horizon=road_vp[1] if road_vp else None; focal=None; focal_status='UNKNOWN'; focal_source='insufficient orthogonal vanishing-point evidence'
    if road_vp and vertical_vp:
        dot=(road_vp[0]-cx)*(vertical_vp[0]-cx)+(road_vp[1]-cy)*(vertical_vp[1]-cy)
        if dot<0: focal=float(np.sqrt(-dot)); focal_status='ESTIMATED'; focal_source='orthogonal road-direction and vertical vanishing points'
    return {'fx':{'value':focal,'unit':'px','status':focal_status,'confidence':.55 if focal else 0.,'uncertainty':.25*focal if focal else None,'source':focal_source},'fy':{'value':focal,'unit':'px','status':focal_status,'confidence':.55 if focal else 0.,'uncertainty':.25*focal if focal else None,'source':focal_source},'cx':{'value':cx,'unit':'px','status':'ASSUMED','confidence':.2,'uncertainty':w*.05,'source':'principal-point image-center prior'},'cy':{'value':cy,'unit':'px','status':'ASSUMED','confidence':.2,'uncertainty':h*.05,'source':'principal-point image-center prior'},'focal_length':{'value':focal,'unit':'px','status':focal_status,'confidence':.55 if focal else 0.,'uncertainty':.25*focal if focal else None,'source':focal_source},'horizontal_fov_deg':{'value':float(np.degrees(2*np.arctan(w/(2*focal)))) if focal else None,'unit':'deg','status':'DERIVED' if focal else 'UNKNOWN','confidence':.5 if focal else 0.,'uncertainty':None,'source':'estimated focal length'},'horizon_y':{'value':horizon,'unit':'px','status':'ESTIMATED' if horizon else 'UNKNOWN','confidence':.45 if horizon else 0.,'uncertainty':h*.04 if horizon else None,'source':'road-line vanishing point'},'vanishing_points':{'road':road_vp,'vertical':vertical_vp,'unit':'px','status':'MEASURED' if road_vp else 'UNKNOWN','confidence':.45 if road_vp else 0.,'uncertainty':None,'source':'Hough line intersections'},'pitch':{'value':None,'unit':'rad','status':'UNKNOWN','confidence':0.,'uncertainty':None,'source':'camera height/ground plane not independently observable'},'roll':{'value':None,'unit':'rad','status':'UNKNOWN','confidence':0.,'uncertainty':None,'source':'insufficient horizon constraints'},'camera_height':{'value':None,'unit':'m','status':'UNKNOWN','confidence':0.,'uncertainty':None,'source':'no metric mounting information provided'},'distortion':{'value':None,'unit':'coefficients','status':'UNKNOWN','confidence':0.,'uncertainty':None,'source':'not recoverable reliably from arbitrary dashcam video'}}

def calibration_matrix(calibration):
    fx=calibration['fx']['value']; fy=calibration['fy']['value']; cx=calibration['cx']['value']; cy=calibration['cy']['value']
    if not all(v is not None and np.isfinite(v) and v>0 for v in (fx,fy,cx,cy)): return None
    return np.array([[fx,0,cx],[0,fy,cy],[0,0,1.]],dtype=np.float64)

def project_relative_xyz(x,y,depth,calibration):
    K=calibration_matrix(calibration)
    if K is None or not np.isfinite(depth): return None
    z=1.0/(float(depth)+1e-3); return {'x':float((x-K[0,2])*z/K[0,0]),'y':float((y-K[1,2])*z/K[1,1]),'z':z,'unit':'relative_camera_units','status':'DERIVED','source':'relative depth + estimated camera matrix'}
print('Calibration is evidence-gated; Hough segment shapes are normalized across OpenCV versions.')

In [ ]:
# Cell 4: multi-lane evidence, temporal prediction, estimated boundaries, and relative 3D geometry
lane_track_state={}; next_lane_id=1; MAX_LANE_MISSES=8

def _marking_components(mask,y):
    row=(mask[y].astype(np.uint8)*255); count,_,stats,cent=cv2.connectedComponentsWithStats(row,8)
    return [float(cent[i,0]) for i in range(1,count) if stats[i,cv2.CC_STAT_AREA]>=2]

def _fit_lane(points,h,w):
    if len(points)<5: return None
    y=np.asarray([p[0] for p in points],float)/h; x=np.asarray([p[1] for p in points],float)/w
    try:
        coef=np.polyfit(y,x,2); residual=np.abs(x-np.polyval(coef,y)); keep=residual<max(.018,2*np.median(residual)+1e-6)
        if int(keep.sum())<5: return None
        filtered=[points[i] for i,k in enumerate(keep) if k]; y=np.asarray([p[0] for p in filtered],float)/h; x=np.asarray([p[1] for p in filtered],float)/w; coef=np.polyfit(y,x,2)
        return filtered,coef,float(max(2,np.std(residual)*w))
    except (ValueError,np.linalg.LinAlgError): return None

def lane_points(frame,road_mask,depth,calibration):
    global next_lane_id,lane_track_state
    h,w=frame.shape[:2]; hsv=cv2.cvtColor(frame,cv2.COLOR_BGR2HSV); bright=(hsv[:,:,2]>145)&(hsv[:,:,1]<105); yellow=(hsv[:,:,0]>=12)&(hsv[:,:,0]<=42)&(hsv[:,:,1]>55)&(hsv[:,:,2]>95); evidence=(bright|yellow)&(road_mask>0)
    ys=np.linspace(.52*h,.97*h,24).astype(int); tracks=[]
    for y in ys:
        xs=_marking_components(evidence,y); used=set()
        for track in tracks:
            pred=track[-1][1]+(track[-1][1]-track[-2][1] if len(track)>1 else 0); choices=[(abs(x-pred),j,x) for j,x in enumerate(xs) if j not in used]
            if choices and min(choices)[0]<max(20,.045*w): _,j,x=min(choices); track.append((int(y),float(x))); used.add(j)
        for j,x in enumerate(xs):
            if j not in used: tracks.append([(int(y),float(x))])
    observed=[]
    for points in tracks:
        fitted=_fit_lane(points,h,w)
        if fitted is None: continue
        pts,coef,unc=fitted; bottom=float(np.polyval(coef,.96)*w); observed.append((bottom,pts,coef,unc))
    lanes=[]; assigned=set()
    for lane_id,state in list(lane_track_state.items()):
        predicted=state['bottom_px']+state.get('velocity_px',0); choices=[(abs(bottom-predicted),i) for i,(bottom,_,_,_) in enumerate(observed) if i not in assigned]
        if choices and min(choices)[0]<.11*w:
            _,i=min(choices); assigned.add(i); bottom,pts,coef,unc=observed[i]; state.update({'bottom_px':bottom,'velocity_px':bottom-state['bottom_px'],'coef':coef.tolist(),'points':pts,'misses':0,'last_status':'MEASURED'})
        else:
            state['bottom_px']=predicted; state['misses']+=1; state['last_status']='ESTIMATED'
    for i,(bottom,pts,coef,unc) in enumerate(observed):
        if i in assigned: continue
        lane_track_state[next_lane_id]={'bottom_px':bottom,'velocity_px':0.0,'coef':coef.tolist(),'points':pts,'misses':0,'last_status':'MEASURED'}; next_lane_id+=1
    for lane_id,state in list(lane_track_state.items()):
        if state['misses']>MAX_LANE_MISSES: del lane_track_state[lane_id]; continue
        coef=np.asarray(state['coef'],float); status=state['last_status']; pts=state['points']; xyz=[]
        if status=='ESTIMATED':
            pts=[(int(y),float(np.polyval(coef,y/h)*w)) for y in ys]
        for yy,xx in pts:
            p=project_relative_xyz(xx,yy,float(depth[min(h-1,yy),min(w-1,max(0,int(xx)))]),calibration); xyz.append({'x':p['x'],'y':p['y'],'z':p['z'],'unit':p['unit'],'status':'DERIVED'} if p else None)
        lanes.append({'lane_id':int(lane_id),'points_px':[(int(x),int(y)) for y,x in pts],'xyz':xyz,'fit_coeff_px_norm':coef.tolist(),'bottom_x_px':float(state['bottom_px']),'status':'MEASURED' if status=='MEASURED' else 'ESTIMATED','metric_status':'DERIVED' if all(p is not None for p in xyz) else 'UNKNOWN','confidence':float(.7 if status=='MEASURED' else max(.2,.6-.08*state['misses'])),'uncertainty_px':float(unc if status=='MEASURED' else max(unc,8+3*state['misses'])),'source':'observed markings + road mask + temporal prediction + relative depth'})
    return sorted(lanes,key=lambda z:z['bottom_x_px'])

def lane_rows(frame_id,timestamp,lanes):
    rows=[]
    for lane in lanes:
        for index,(x,y) in enumerate(lane['points_px']):
            p=lane['xyz'][index] if index<len(lane['xyz']) else None; rows.append({'frame':frame_id,'timestamp_s':timestamp,'lane_id':lane['lane_id'],'boundary':'observed' if lane['status']=='MEASURED' else 'predicted','x_px':x,'y_px':y,'x_3d':p['x'] if p else None,'y_3d':p['y'] if p else None,'z_3d':p['z'] if p else None,'unit_3d':p['unit'] if p else 'unknown','status':lane['status'],'metric_status':lane['metric_status'],'confidence':lane['confidence'],'uncertainty_px':lane['uncertainty_px'],'source':lane['source']})
    return rows

def derive_lane_widths(lanes):
    out=[]
    for left,right in zip(lanes[:-1],lanes[1:]):
        pairs=[(a,b) for a,b in zip(left['xyz'],right['xyz']) if a is not None and b is not None]
        if len(pairs)<3: continue
        values=[float(np.linalg.norm(np.array([a['x'],a['y'],a['z']])-np.array([b['x'],b['y'],b['z']]))) for a,b in pairs]; status='DERIVED' if left['status']=='MEASURED' and right['status']=='MEASURED' else 'ESTIMATED'
        out.append({'left_lane_id':left['lane_id'],'right_lane_id':right['lane_id'],'width':float(np.median(values)),'unit':'relative_camera_units','status':status,'confidence':float(min(left['confidence'],right['confidence'])*.5),'uncertainty':float(np.std(values)),'source':'adjacent temporal 3D lane tracks'})
    return out
print('Lane estimates persist through short occlusions; estimated boundaries are separated from observed boundaries and never treated as measured.')

In [ ]:
# Cell 5: robust monocular VO, temporal fusion, explicit speed semantics, and scale provenance
from scipy.signal import savgol_filter

def vo_step(prev,cur,K):
    if K is None: return None
    a=cv2.cvtColor(prev,cv2.COLOR_BGR2GRAY); b=cv2.cvtColor(cur,cv2.COLOR_BGR2GRAY); p=cv2.goodFeaturesToTrack(a,maxCorners=1800,qualityLevel=.01,minDistance=7,blockSize=7)
    if p is None or len(p)<30: return None
    q,st,_=cv2.calcOpticalFlowPyrLK(a,b,p,None,winSize=(21,21),maxLevel=3,criteria=(cv2.TERM_CRITERIA_EPS|cv2.TERM_CRITERIA_COUNT,30,.01)); good=st.reshape(-1).astype(bool); p,q=p[good],q[good]
    if len(p)<20: return None
    back,st2,_=cv2.calcOpticalFlowPyrLK(b,a,q,None,winSize=(21,21),maxLevel=3); good=st2.reshape(-1).astype(bool); fb=np.linalg.norm(p[good].reshape(-1,2)-back[good].reshape(-1,2),axis=1); keep=good.copy(); keep[good]=fb<1.5; p,q=p[keep],q[keep]
    if len(p)<15: return None
    E,mask=cv2.findEssentialMat(p,q,K,method=cv2.RANSAC,prob=.999,threshold=1.0)
    if E is None or mask is None or int(mask.sum())<12: return None
    _,R,t,pose_mask=cv2.recoverPose(E,p,q,K,mask=mask); return R,t.reshape(3),int(mask.sum()),int(pose_mask.sum()),len(p)

def _robust_smooth(values):
    values=np.asarray(values,float); valid=np.isfinite(values); out=values.copy()
    if valid.sum()>=5:
        med=np.nanmedian(values); mad=np.nanmedian(np.abs(values[valid]-med))+1e-6; out[np.abs(values-med)>6*mad]=np.nan; series=pd.Series(out).interpolate(limit_direction='both').to_numpy()
        if len(series)>=7: series=savgol_filter(series,7,2,mode='interp')
        return series
    return out

def process_video():
    global calibration
    cap=cv2.VideoCapture(INPUT_VIDEO); frames=[]; lane_rows_all=[]; poses=[]; prev=None; pose=np.eye(4); frame_i=processed=failed_vo=vo_success=0; calibration=None; started=perf_counter()
    while True:
        ok,frame=cap.read()
        if not ok or (MAX_FRAMES and processed>=MAX_FRAMES): break
        if frame_i%FRAME_STRIDE: frame_i+=1; continue
        depth,_=infer_depth(frame); road,_=infer_road_mask(frame)
        if calibration is None: calibration=estimate_calibration(frame,road)
        lanes=lane_points(frame,road,depth,calibration); lane_rows_all.extend(lane_rows(frame_i,frame_i/FPS,lanes)); displacement=None; inliers=None; rotation_rad=None; vo_status='INITIAL'; K=calibration_matrix(calibration)
        if prev is not None:
            step=vo_step(prev,frame,K)
            if step is None: failed_vo+=1; vo_status='FAILED'
            else:
                R,t,inliers,pose_inliers,tracked=step; T=np.eye(4); T[:3,:3]=R; T[:3,3]=t; pose=pose@np.linalg.inv(T); displacement=float(np.linalg.norm(t)); rotation_rad=float(np.arccos(np.clip((np.trace(R)-1)/2,-1,1))); vo_success+=1; vo_status='MEASURED'
        poses.append({'frame':frame_i,'timestamp_s':frame_i/FPS,'x_relative':float(pose[0,3]),'y_relative':float(pose[1,3]),'z_relative':float(pose[2,3]),'frame_displacement_relative':displacement,'rotation_rad':rotation_rad,'vo_inliers':inliers,'vo_status':vo_status,'status':'DERIVED','metric_status':'UNKNOWN'}); frames.append((frame_i,frame,road,lanes,depth)); prev=frame; frame_i+=1; processed+=1
    cap.release()
    if not processed: raise RuntimeError('No frames were processed')
    elapsed=max(perf_counter()-started,1e-6); raw=np.asarray([p['frame_displacement_relative']*FPS if p['frame_displacement_relative'] is not None else np.nan for p in poses]); speed=_robust_smooth(raw); accel=np.gradient(speed,1/max(FPS,1e-6)) if np.isfinite(speed).sum()>1 else np.full(len(speed),np.nan); accel_limit=float(os.environ.get('MAX_RELATIVE_ACCELERATION','12.0')); accel_quality=np.where(np.isfinite(accel)&(np.abs(accel)>accel_limit),'IMPLAUSIBLE','OK')
    scale=METRIC_SCALE; scale_status='ASSUMED' if scale is not None else 'UNKNOWN'; metric_status='ASSUMED_SCALE_DERIVED' if scale_status=='ASSUMED' else 'UNKNOWN'; scale_source='explicit user METRIC_SCALE_M_PER_RELATIVE_UNIT' if scale is not None else 'monocular scale ambiguity; no external metric reference'
    for i,p in enumerate(poses):
        p['speed_relative_units_s']=float(speed[i]) if np.isfinite(speed[i]) else None; p['acceleration_relative_units_s2']=float(accel[i]) if np.isfinite(accel[i]) else None; p['acceleration_quality']=str(accel_quality[i]); p['speed_mps']=float(speed[i]*scale) if scale is not None and np.isfinite(speed[i]) else None; p['speed_status']=metric_status if scale is not None else 'RELATIVE'; p['scale_source']=scale_source
    scale_info={'value':scale,'unit':'m per relative unit','status':scale_status,'metric_result_status':metric_status,'confidence':0.8 if scale is not None else 0.0,'uncertainty':None,'source':scale_source}
    pd.DataFrame(poses).to_csv(ROOT/'ego_trajectory.csv',index=False); pd.DataFrame([{'frame':p['frame'],'timestamp_s':p['timestamp_s'],'speed_relative_units_s':p['speed_relative_units_s'],'speed_mps':p['speed_mps'],'speed_status':p['speed_status'],'acceleration_relative_units_s2':p['acceleration_relative_units_s2'],'acceleration_quality':p['acceleration_quality'],'scale_source':p['scale_source']} for p in poses]).to_csv(ROOT/'speed_profile.csv',index=False)
    print({'processed_frames':processed,'frames_skipped':max(0,TOTAL-processed),'vo_success':vo_success,'vo_failures':failed_vo,'vo_status':'PASS' if vo_success else 'UNAVAILABLE','scale':scale_info,'implausible_acceleration_count':int((accel_quality=='IMPLAUSIBLE').sum()),'processing_seconds':elapsed,'processed_fps':processed/elapsed})
    return frames,lane_rows_all,poses,calibration,processed,failed_vo,scale_info,elapsed
frames,lane_rows_all,poses,calibration,processed,failed_vo,scale_info,processing_seconds=process_video()

In [ ]:
# Cell 6: YOLO11m, temporal tracking with missed-frame prediction, depth-backed 3D localization
active={}; next_id=1; vehicle_rows=[]; track_history={}; MAX_TRACK_MISSES=5

def box_iou(a,b):
    x1=max(a[0],b[0]); y1=max(a[1],b[1]); x2=min(a[2],b[2]); y2=min(a[3],b[3]); inter=max(0,x2-x1)*max(0,y2-y1); union=max(1,(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter); return inter/union

def associate_lane(x,lane_list):
    ordered=sorted(lane_list,key=lambda z:z['bottom_x_px'])
    if not ordered: return None
    if x<ordered[0]['bottom_x_px']: return f'left_of_{ordered[0]["lane_id"]}'
    for left,right in zip(ordered[:-1],ordered[1:]):
        if left['bottom_x_px']<=x<=right['bottom_x_px']: return f'between_{left["lane_id"]}_{right["lane_id"]}'
    return f'right_of_{ordered[-1]["lane_id"]}'

def append_vehicle(frame_i,now,tid,det,depth,lanes,status='MEASURED'):
    bb=det['bbox']; cx=float((bb[0]+bb[2])/2); cy=float(bb[3]); yi=min(HEIGHT-1,max(0,int(cy))); xi=min(WIDTH-1,max(0,int(cx))); point=project_relative_xyz(cx,cy,float(depth[yi,xi]),calibration); hist=track_history.setdefault(tid,[]); rel_vx=rel_vy=rel_vz=ttc=None
    if point and hist:
        prev=hist[-1]; delta=max(now-prev['timestamp_s'],1/FPS); rel_vx=(point['x']-prev['x'])/delta; rel_vy=(point['y']-prev['y'])/delta; rel_vz=(point['z']-prev['z'])/delta; closing=-rel_vz; ttc=point['z']/closing if closing>1e-6 and point['z']>0 else None
    if point: hist.append({'timestamp_s':now,'x':point['x'],'y':point['y'],'z':point['z']}); hist[:]=hist[-30:]
    scale=scale_info.get('value'); metric_distance=float(np.linalg.norm([point['x'],point['y'],point['z']])*scale) if point and scale else None; metric_v=float(np.linalg.norm([rel_vx or 0,rel_vy or 0,rel_vz or 0])*scale) if scale and rel_vx is not None else None; metric_status=scale_info.get('metric_result_status','UNKNOWN') if scale and point else 'UNKNOWN'
    vehicle_rows.append({'frame':frame_i,'timestamp_s':now,'track_id':tid,'class':det['class'],'confidence':det['confidence'],'x1':bb[0],'y1':bb[1],'x2':bb[2],'y2':bb[3],'x_3d':point['x'] if point else None,'y_3d':point['y'] if point else None,'z_3d':point['z'] if point else None,'distance_relative':float(np.linalg.norm([point['x'],point['y'],point['z']])) if point else None,'distance_m':metric_distance,'relative_velocity_x':rel_vx,'relative_velocity_y':rel_vy,'relative_velocity_z':rel_vz,'relative_velocity_units_s':float(np.linalg.norm([rel_vx or 0,rel_vy or 0,rel_vz or 0])) if rel_vx is not None else None,'absolute_velocity_mps':metric_v,'lane_id':associate_lane(cx,lanes),'ttc_s':ttc,'status':status if point else 'UNKNOWN','metric_status':metric_status,'source':'YOLO11m + temporal IoU prediction + relative depth + camera projection'})

for frame_i,frame,road,lanes,depth in tqdm(frames,desc='vehicles'):
    now=frame_i/FPS; result=VEHICLE_MODEL.predict(frame,device=0,half=HALF,verbose=False)[0]; detections=[]
    for box,cls,conf in zip(result.boxes.xyxy.cpu().numpy(),result.boxes.cls.cpu().numpy(),result.boxes.conf.cpu().numpy()):
        name=VEHICLE_MODEL.names[int(cls)]
        if name in COCO_VEHICLES: detections.append({'bbox':box.tolist(),'class':name,'confidence':float(conf)})
    matched=set(); seen=set()
    for det in detections:
        choices=[(box_iou(det['bbox'],v['bbox']),tid) for tid,v in active.items() if tid not in matched and frame_i-v['frame']<=MAX_TRACK_MISSES]; best=max(choices,default=(0,None)); tid=best[1] if best[0]>=.25 else next_id
        if tid==next_id: next_id+=1
        matched.add(tid); seen.add(tid); active[tid]={'bbox':det['bbox'],'frame':frame_i,'class':det['class'],'misses':0}; append_vehicle(frame_i,now,tid,det,depth,lanes,'MEASURED')
    for tid,state in list(active.items()):
        if tid in seen: continue
        state['misses']+=1
        if state['misses']>MAX_TRACK_MISSES: del active[tid]; continue
        det={'bbox':state['bbox'],'class':state['class'],'confidence':0.0}; append_vehicle(frame_i,now,tid,det,depth,lanes,'ESTIMATED')
pd.DataFrame(vehicle_rows).to_csv(ROOT/'vehicle_tracks.csv',index=False)
print({'vehicle_rows':len(vehicle_rows),'unique_tracks':len(track_history),'tracking_status':'PASS' if vehicle_rows else 'UNAVAILABLE','predicted_rows':sum(r['status']=='ESTIMATED' for r in vehicle_rows),'3d_status':'PASS' if any(r['x_3d'] is not None for r in vehicle_rows) else 'UNAVAILABLE','metric_scale':scale_info})

In [ ]:
# Cell 7: road reconstruction, lane summary, topology evidence, BEV, and 3D visualization
lane_df=pd.DataFrame(lane_rows_all); ego_df=pd.DataFrame(poses); centerline_rows=[]; road_rows=[]; lane_width_rows=[]
for frame_i,frame,road,lanes,depth in frames:
    ordered=sorted(lanes,key=lambda z:z['bottom_x_px']); lane_width_rows.extend([dict(x,frame=frame_i,timestamp_s=frame_i/FPS) for x in derive_lane_widths(ordered)])
    for lane in ordered:
        xyz=[p for p in lane['xyz'] if p is not None]
        for p in xyz: road_rows.append({'frame':frame_i,'timestamp_s':frame_i/FPS,'geometry_type':'lane_boundary','lane_id':lane['lane_id'],'x':p['x'],'y':p['y'],'z':p['z'],'unit':p['unit'],'status':lane['status'],'source':lane['source']})
        if len(xyz)>=3:
            dz=np.diff([p['z'] for p in xyz]); dx=np.diff([p['x'] for p in xyz]); heading=float(np.arctan2(np.nanmedian(dx),np.nanmedian(dz))); curvature=float(np.nanmedian(np.abs(np.diff(dx)))) if len(dx)>1 else None; slope=float(np.nanmedian(np.diff([p['y'] for p in xyz])/(np.abs(dz)+1e-6))) if len(dz) else None
            for p in xyz: lane_rows_all.append({'frame':frame_i,'timestamp_s':frame_i/FPS,'lane_id':lane['lane_id'],'boundary':'summary','x_3d':p['x'],'y_3d':p['y'],'z_3d':p['z'],'heading_rad':heading,'curvature_relative':curvature,'slope_relative':slope,'length_relative':float(np.ptp([q['z'] for q in xyz])),'status':lane['status'],'metric_status':lane['metric_status'],'confidence':lane['confidence'],'uncertainty_px':lane['uncertainty_px'],'source':'temporal lane geometry summary'})
    for left,right in zip(ordered[:-1],ordered[1:]):
        for a,b in zip(left['xyz'],right['xyz']):
            if a is not None and b is not None: centerline_rows.append({'frame':frame_i,'timestamp_s':frame_i/FPS,'lane_id':f'{left["lane_id"]}_{right["lane_id"]}','x':(a['x']+b['x'])/2,'y':(a['y']+b['y'])/2,'z':(a['z']+b['z'])/2,'unit':'relative_camera_units','status':'DERIVED','source':'midpoint of adjacent temporal 3D boundaries'})
for p in poses: road_rows.append({'frame':p['frame'],'timestamp_s':p['timestamp_s'],'geometry_type':'ego_trajectory','lane_id':None,'x':p['x_relative'],'y':p['y_relative'],'z':p['z_relative'],'unit':'relative_vo_units','status':'DERIVED','source':'monocular essential-matrix VO'})
if lane_df.empty: lane_df=pd.DataFrame([{'status':'UNAVAILABLE','reason':'no lane marking evidence passed quality gate'}])
for col in ['lane_id','x_3d','y_3d','z_3d']:
    if col not in lane_df: lane_df[col]=np.nan
pd.DataFrame(road_rows).to_csv(ROOT/'road_geometry.csv',index=False); lane_df.to_csv(ROOT/'lane_geometry.csv',index=False)
frame_counts=[len(lanes) for _,_,_,lanes,_ in frames]; best_count=max(frame_counts,default=0); stable_count=int(round(np.percentile(frame_counts,75))) if frame_counts else 0; best_count=max(best_count,stable_count); status_series=lane_df['status'] if 'status' in lane_df else pd.Series('',index=lane_df.index); observed_ids=sorted(set(lane_df.loc[status_series=='MEASURED','lane_id'].dropna().astype(int))) if 'lane_id' in lane_df else []; ego_lane='unknown'; adjacent={'left':None,'right':None}
if observed_ids:
    ego_lane=int(min(observed_ids,key=lambda i:abs(float(lane_df[lane_df.lane_id==i].x_px.mean()-WIDTH/2))) if 'x_px' in lane_df else observed_ids[len(observed_ids)//2]); adjacent={'left':observed_ids[observed_ids.index(ego_lane)-1] if observed_ids.index(ego_lane)>0 else None,'right':observed_ids[observed_ids.index(ego_lane)+1] if observed_ids.index(ego_lane)<len(observed_ids)-1 else None}
valid_z=lane_df['z_3d'].dropna(); visible_range={'near':float(valid_z.min()) if len(valid_z) else None,'far':float(valid_z.max()) if len(valid_z) else None,'unit':'relative_camera_units','status':'DERIVED' if len(valid_z) else 'UNKNOWN','confidence':0.25 if len(valid_z) else 0.0,'uncertainty':None,'source':'observed and temporally estimated lane depth range'}; traj=ego_df[['x_relative','z_relative']].to_numpy(float); step_dist=np.linalg.norm(np.diff(traj,axis=0),axis=1) if len(traj)>1 else np.array([])
road_metrics={'visible_reconstructed_road_range':visible_range,'visible_reconstructed_road_length_relative':float(np.ptp(valid_z)) if len(valid_z) else None,'total_dashcam_distance_relative':float(step_dist.sum()) if len(step_dist) else None,'total_dashcam_distance_metric':float(step_dist.sum()*scale_info['value']) if scale_info.get('value') else None,'lane_count_best_supported':best_count,'lane_count_observed_max':max(frame_counts,default=0),'ego_lane':ego_lane,'adjacent_lanes':adjacent,'lane_widths':lane_width_rows,'road_width':{'value':None,'unit':'relative_camera_units','status':'UNKNOWN','confidence':0.0,'uncertainty':None,'source':'outer road edges not independently observed'},'scale':scale_info,'status':'PARTIAL'}; (ROOT/'road_metrics.json').write_text(json.dumps(road_metrics,indent=2,default=str))
counts=frame_counts; events=[]
for i in range(1,len(counts)):
    if counts[i]!=counts[i-1] and i>=2 and counts[i]==counts[i-2]: events.append({'frame':frames[i][0],'timestamp_s':frames[i][0]/FPS,'type':'lane_addition' if counts[i]>counts[i-1] else 'lane_drop','from_boundary_count':counts[i-1],'to_boundary_count':counts[i],'status':'ESTIMATED','confidence':0.3,'uncertainty':'visibility and prediction uncertainty','source':'persistent lane-boundary count change'})
links=[]; all_ids=sorted(set(lane_df['lane_id'].dropna().astype(int))) if 'lane_id' in lane_df else []
for a,b in zip(all_ids[:-1],all_ids[1:]): links.append({'from_lane_id':int(a),'to_lane_id':int(b),'relation':'adjacent_observed_or_estimated','status':'DERIVED','confidence':0.3,'source':'co-observed ordered lane boundaries'})
topology={'events':events,'lane_connectivity':links,'road_connectivity':[],'status':'ESTIMATED' if events else 'UNKNOWN','reason':'only persistent geometric evidence is emitted; unsupported junction classes remain UNKNOWN'}; (ROOT/'topology.json').write_text(json.dumps(topology,indent=2))
import matplotlib.pyplot as plt
plt.figure(figsize=(10,7))
for lid,g in lane_df.dropna(subset=['x_3d','z_3d']).groupby('lane_id'): plt.plot(g.x_3d,g.z_3d,'.-',label=f'boundary {int(lid)}')
if centerline_rows:
    cdf=pd.DataFrame(centerline_rows)
    for lid,g in cdf.groupby('lane_id'): plt.plot(g.x,g.z,'--',label=f'center {lid}')
plt.plot(ego_df.x_relative,ego_df.z_relative,'k-',linewidth=2,label='ego VO'); vehicle_df=pd.DataFrame(vehicle_rows)
if not vehicle_df.empty and all(c in vehicle_df for c in ['x_3d','z_3d']):
    for tid,g in vehicle_df.dropna(subset=['x_3d','z_3d']).groupby('track_id'): plt.plot(g.x_3d,g.z_3d,'r:',label=f'object {tid}')
plt.xlabel('relative lateral X'); plt.ylabel('relative forward Z'); plt.title('Reconstructed relative BEV'); plt.grid(); plt.legend(loc='best',fontsize=7); plt.tight_layout(); plt.savefig(ROOT/'BEV.png',dpi=160); plt.close()
fig=plt.figure(figsize=(10,7)); ax=fig.add_subplot(111,projection='3d')
for lid,g in lane_df.dropna(subset=['x_3d','y_3d','z_3d']).groupby('lane_id'): ax.plot(g.x_3d,g.y_3d,g.z_3d,'.-',label=f'boundary {int(lid)}')
ax.plot(ego_df.x_relative,ego_df.y_relative,ego_df.z_relative,'k-',linewidth=2,label='ego VO')
if not vehicle_df.empty and all(c in vehicle_df for c in ['x_3d','y_3d','z_3d']):
    for tid,g in vehicle_df.dropna(subset=['x_3d','y_3d','z_3d']).groupby('track_id'): ax.plot(g.x_3d,g.y_3d,g.z_3d,'r.',label=f'object {tid}')
ax.set_xlabel('X relative'); ax.set_ylabel('Y relative'); ax.set_zlabel('Z relative'); ax.set_title('Reconstructed 3D scene'); ax.legend(fontsize=7); fig.tight_layout(); fig.savefig(ROOT/'scene_3d.png',dpi=160); plt.close()
print({'road_rows':len(road_rows),'best_supported_lane_count':best_count,'ego_lane':ego_lane,'topology_status':topology['status'],'bev_bytes':(ROOT/'BEV.png').stat().st_size,'scene_3d_bytes':(ROOT/'scene_3d.png').stat().st_size})

In [ ]:
# Cell 8: evidence-derived OpenDRIVE polyline, supported lanes only, and structural checks
from lxml import etree

def finite_rows(rows,keys): return all(all(np.isfinite(float(r[k])) for k in keys) for r in rows)
path_rows=[{'x':float(p['x_relative']),'y':float(p['z_relative']),'z':float(p['y_relative'])} for p in poses]; segments=[]; road_len=0.0
for a,b in zip(path_rows[:-1],path_rows[1:]):
    length=float(np.hypot(b['x']-a['x'],b['y']-a['y']))
    if length>1e-6: segments.append({'s':road_len,'x':a['x'],'y':a['y'],'hdg':float(np.arctan2(b['y']-a['y'],b['x']-a['x'])),'length':length,'status':'DERIVED','source':'monocular VO trajectory'}); road_len+=length
root=etree.Element('OpenDRIVE'); etree.SubElement(root,'header',revMajor='1',revMinor='6',name='monocular_reconstruction_relative',version='1.00',date='2026-09-10',north='0',south='0',east='0',west='0'); road=etree.SubElement(root,'road',name='reconstructed_relative_road',length=f'{road_len:.9f}',id='1',junction='-1'); pv=etree.SubElement(road,'planView')
for g in segments:
    geom=etree.SubElement(pv,'geometry',s=f"{g['s']:.9f}",x=f"{g['x']:.9f}",y=f"{g['y']:.9f}",hdg=f"{g['hdg']:.9f}",length=f"{g['length']:.9f}"); etree.SubElement(geom,'line')
elev=etree.SubElement(road,'elevationProfile')
if path_rows: etree.SubElement(elev,'elevation',s='0',a=f"{path_rows[0]['z']:.9f}",b='0',c='0',d='0')
lanes_node=etree.SubElement(road,'lanes'); section=etree.SubElement(lanes_node,'laneSection',s='0'); etree.SubElement(section,'left'); center=etree.SubElement(section,'center'); right=etree.SubElement(section,'right'); etree.SubElement(center,'lane',id='0',type='none',level='false')
width_values=[float(x['width']) for x in lane_width_rows if x.get('width') is not None and np.isfinite(x['width']) and x['width']>0]; supported_lane_ids=sorted({int(x) for x in lane_df['lane_id'].dropna()}) if 'lane_id' in lane_df else []
# Only lanes represented by actual/estimated reconstructed boundaries are exported.
for index,lane_id in enumerate(supported_lane_ids[:-1],1):
    lane=etree.SubElement(right,'lane',id=str(-index),type='driving',level='false')
    widths=[x['width'] for x in lane_width_rows if x.get('left_lane_id')==lane_id and x.get('width') is not None and np.isfinite(x['width']) and x['width']>0]
    if widths: etree.SubElement(lane,'width',sOffset='0',a=f"{float(np.median(widths)):.9f}",b='0',c='0',d='0')
    etree.SubElement(lane,'roadMark',sOffset='0',type='broken',weight='standard',color='standard',width='0.1',laneChange='both')
xodr=etree.tostring(root,pretty_print=True,xml_declaration=True,encoding='UTF-8'); (ROOT/'road.xodr').write_bytes(xodr)
checks=[]
def odr_check(name,condition,detail=''): checks.append({'check':name,'status':'PASS' if condition else 'FAIL','detail':detail})
try: etree.fromstring(xodr); odr_check('xml_well_formed',True,'lxml parsed generated XML')
except Exception as exc: odr_check('xml_well_formed',False,str(exc))
odr_check('road_length_positive',road_len>0,str(road_len)); odr_check('geometry_segments_present',len(segments)>0,str(len(segments))); starts=[g['s'] for g in segments]; ends=[g['s']+g['length'] for g in segments]; odr_check('s_coordinate_continuity',all(abs(starts[i+1]-ends[i])<1e-6 for i in range(len(segments)-1)),'polyline segment boundaries'); odr_check('geometry_finite',finite_rows(segments,['s','x','y','hdg','length']) if segments else False,'no NaN/Inf geometry'); odr_check('lane_ids_valid',all(int(x.get('id'))!=0 for x in right.findall('lane')),'center lane remains id 0'); odr_check('supported_lane_count_consistent',len(right.findall('lane'))<=max(0,len(supported_lane_ids)-1),'no unsupported nominal lane')
validation={'overall':'PASS' if all(x['status']=='PASS' for x in checks) else ('PARTIAL' if any(x['status']=='PASS' for x in checks) else 'FAIL'),'checks':checks,'geometry_status':'DERIVED relative VO polyline','scale_status':scale_info['status'],'limitations':['coordinates and widths are relative unless explicit scale is supplied','junction links omitted unless reliable road-to-road evidence exists']}; (ROOT/'opendrive_validation_report.json').write_text(json.dumps(validation,indent=2)); print(json.dumps({'road_length_relative':road_len,'segments':len(segments),'supported_lane_ids':supported_lane_ids,'validation':validation},indent=2))

In [ ]:
# Cell 9: annotated video, parallel visual evidence, dashboard report, and processing statistics
fourcc=cv2.VideoWriter_fourcc(*'mp4v'); overlay_path=ROOT/'annotated_adas_video.mp4'; writer=cv2.VideoWriter(str(overlay_path),fourcc,FPS,(WIDTH,HEIGHT))
if not writer.isOpened(): raise RuntimeError('Could not open annotated video writer')
vehicles_by_frame={}; [vehicles_by_frame.setdefault(row['frame'],[]).append(row) for row in vehicle_rows]; pose_by_frame={p['frame']:p for p in poses}; first_panels=None
for frame_i,frame,road,lanes,depth in tqdm(frames,desc='annotated video'):
    original=frame.copy(); overlay=frame.copy(); overlay[road.astype(bool)]=((overlay[road.astype(bool)].astype(np.float32)*.62)+np.array([25,55,0])).astype(np.uint8); vis=cv2.addWeighted(frame,.58,overlay,.42,0); lane_view=vis.copy()
    for lane in lanes:
        pts=np.asarray(lane['points_px'],np.int32).reshape(-1,1,2); color=(0,255,255) if lane['status']=='MEASURED' else (255,160,0)
        if len(pts)>1: cv2.polylines(lane_view,[pts],False,color,3)
        if len(pts): cv2.putText(lane_view,f'lane {lane["lane_id"]} {lane["status"]}',tuple(pts[0,0]),cv2.FONT_HERSHEY_SIMPLEX,.5,color,2)
    vehicle_view=lane_view.copy()
    for row in vehicles_by_frame.get(frame_i,[]):
        p1=(int(row['x1']),int(row['y1'])); p2=(int(row['x2']),int(row['y2'])); cv2.rectangle(vehicle_view,p1,p2,(0,190,0),2); d=f"z={row['distance_relative']:.2f} rel" if row['distance_relative'] is not None else 'z=UNKNOWN'; v=f"v={row['relative_velocity_units_s']:.2f} rel/s" if row['relative_velocity_units_s'] is not None else 'v=UNKNOWN'; cv2.putText(vehicle_view,f"{row['class']} #{row['track_id']} {row['lane_id']} {d} {v}",(p1[0],max(18,p1[1]-6)),cv2.FONT_HERSHEY_SIMPLEX,.42,(0,230,0),2)
    pose=pose_by_frame.get(frame_i,{}); speed=pose.get('speed_mps'); speed_text=f'{speed:.2f} m/s' if speed is not None else (f"{pose.get('speed_relative_units_s'):.2f} relative/s" if pose.get('speed_relative_units_s') is not None else 'UNKNOWN'); cv2.putText(vehicle_view,f'ego speed: {speed_text} | status: {pose.get("speed_status","UNKNOWN")} | scale: {scale_info.get("status")}',(18,28),cv2.FONT_HERSHEY_SIMPLEX,.5,(255,255,255),2); cv2.putText(vehicle_view,f'frame {frame_i} t={frame_i/FPS:.2f}s | ego lane={road_metrics.get("ego_lane","unknown")}',(18,52),cv2.FONT_HERSHEY_SIMPLEX,.5,(255,255,255),2); writer.write(vehicle_view)
    if first_panels is None:
        depth_view=cv2.normalize(depth,None,0,255,cv2.NORM_MINMAX).astype(np.uint8); depth_view=cv2.applyColorMap(depth_view,cv2.COLORMAP_MAGMA); bev_img=cv2.imread(str(ROOT/'BEV.png')) if (ROOT/'BEV.png').exists() else np.zeros_like(frame); scene_img=cv2.imread(str(ROOT/'scene_3d.png')) if (ROOT/'scene_3d.png').exists() else np.zeros_like(frame)
        def fit(img): return cv2.resize(img,(WIDTH,HEIGHT))
        first_panels=[('original',original),('road_lane',lane_view),('vehicles',vehicle_view),('depth_3d',depth_view),('BEV',fit(bev_img)),('scene_3d',fit(scene_img))]
writer.release()
if first_panels:
    labeled=[]
    for label,img in first_panels:
        panel=img.copy(); cv2.rectangle(panel,(0,0),(220,32),(0,0,0),-1); cv2.putText(panel,label,(8,23),cv2.FONT_HERSHEY_SIMPLEX,.7,(255,255,255),2); labeled.append(panel)
    top=cv2.hconcat(labeled[:3]); bottom=cv2.hconcat(labeled[3:]); cv2.imwrite(str(VIS_ROOT/'parallel_frame_evidence.png'),cv2.vconcat([top,bottom]));
calibration_report=calibration; (PROJECT_DIRS['reports']/'calibration.json').write_text(json.dumps(calibration_report,indent=2)); shutil.copy2(ROOT/'calibration.json',PROJECT_DIRS['reports']/'calibration.json') if (ROOT/'calibration.json').exists() else (ROOT/'calibration.json').write_text(json.dumps(calibration_report,indent=2))
vo_success=sum(p['vo_status']=='MEASURED' for p in poses); depth_inferences=processed; lane_observations=len(lane_rows_all); observed_lane_rows=int(sum(x.get('status')=='MEASURED' for x in lane_rows_all)); estimated_lane_rows=int(sum(x.get('status')=='ESTIMATED' for x in lane_rows_all)); predicted_lane_rows=int(sum(x.get('boundary')=='predicted' for x in lane_rows_all));
report={'title':'Monocular ADAS reconstruction report','prototype_status':'production-oriented monocular ADAS reconstruction prototype','input':{'video':INPUT_VIDEO,'fps':FPS,'width':WIDTH,'height':HEIGHT,'frames_total':TOTAL,'frames_processed':processed,'frame_stride':FRAME_STRIDE,'frames_skipped':max(0,TOTAL-processed)},'hardware':{'device':DEVICE,'torch':torch.__version__,'cuda':torch.version.cuda},'performance':{'processing_seconds':processing_seconds,'processed_fps':processed/max(processing_seconds,1e-6)},'models':MODEL_SELECTION,'lane_system':'multi-lane geometric reconstruction from segmentation/marking evidence + depth + temporal tracking','lane_evidence':{'directly_observed_rows':observed_lane_rows,'estimated_rows':estimated_lane_rows,'predicted_temporal_rows':predicted_lane_rows,'best_supported_lane_count':road_metrics.get('lane_count_best_supported',0),'ego_lane':road_metrics.get('ego_lane'),'adjacent_lanes':road_metrics.get('adjacent_lanes')},'calibration':calibration_report,'stages':{'depth':{'status':'PASS' if depth_inferences==processed else 'PARTIAL','inferences':depth_inferences},'lanes':{'status':'PASS' if lane_observations else 'UNAVAILABLE','observations':lane_observations},'vehicles':{'status':'PASS' if vehicle_rows else 'UNAVAILABLE','observations':len(vehicle_rows)},'tracking':{'status':'PASS' if track_history else 'UNAVAILABLE','tracks':len(track_history)},'ego_motion':{'status':'PASS' if vo_success else 'UNAVAILABLE','successful_steps':vo_success,'failed_steps':failed_vo},'scale':scale_info,'speed':{'status':'ASSUMED_SCALE_DERIVED' if scale_info.get('status')=='ASSUMED' else 'RELATIVE','metric_available':scale_info.get('value') is not None},'road_geometry':{'status':'PASS' if road_rows else 'UNAVAILABLE','rows':len(road_rows)},'topology':topology['status'],'opendrive':validation['overall']},'visualizations':{'parallel_frame':str(VIS_ROOT/'parallel_frame_evidence.png'),'annotated_video':str(overlay_path)},'limitations':['relative depth is not meters','absolute monocular scale is unavailable unless explicitly supplied','this is not a dedicated learned 3D lane detector','junction classes remain UNKNOWN without reliable geometric evidence']}
(ROOT/'calibration.json').write_text(json.dumps(calibration_report,indent=2)); (ROOT/'processing_report.json').write_text(json.dumps({'model_selection':MODEL_SELECTION,'report':report},indent=2,default=str)); html='<html><head><meta charset="utf-8"><title>ADAS reconstruction dashboard</title><style>body{font-family:Arial;max-width:1200px;margin:2rem auto}img{max-width:100%;border:1px solid #ccc}pre{white-space:pre-wrap;background:#f5f5f5;padding:1rem}</style></head><body><h1>Monocular ADAS Reconstruction Dashboard</h1><h2>Status: production-oriented research prototype</h2><pre>'+json.dumps(report,indent=2,default=str)+'</pre><h2>Observed / estimated / predicted lane evidence</h2><p>Directly observed: '+str(observed_lane_rows)+' rows. Estimated: '+str(estimated_lane_rows)+' rows. Predicted temporal: '+str(predicted_lane_rows)+' rows.</p><img src="visualizations/parallel_frame_evidence.png"><h2>BEV</h2><img src="BEV.png"><h2>3D scene</h2><img src="scene_3d.png"></body></html>'; (ROOT/'final_report.html').write_text(html); print({'annotated_video_bytes':overlay_path.stat().st_size,'parallel_visualization':str(VIS_ROOT/'parallel_frame_evidence.png'),'lane_evidence':report['lane_evidence'],'report':str(ROOT/'final_report.html')})

In [ ]:
# Cell 10: content, semantic, UI-output, consistency, and anti-fabrication validation
required=['annotated_adas_video.mp4','lane_geometry.csv','vehicle_tracks.csv','ego_trajectory.csv','speed_profile.csv','road_metrics.json','calibration.json','topology.json','road_geometry.csv','BEV.png','scene_3d.png','processing_report.json','final_report.html']; checks={}
def add(name,status,detail): checks[name]={'status':status,'detail':detail}
for name in required:
    path=ROOT/name; add(f'artifact:{name}','PASS' if path.exists() and path.stat().st_size>0 else 'FAIL',f'bytes={path.stat().st_size if path.exists() else 0}')
add('artifact:parallel_frame_evidence.png','PASS' if (VIS_ROOT/'parallel_frame_evidence.png').exists() and (VIS_ROOT/'parallel_frame_evidence.png').stat().st_size>0 else 'FAIL','visualization saved under outputs/visualizations')
for name in ['lane_geometry.csv','vehicle_tracks.csv','ego_trajectory.csv','speed_profile.csv','road_geometry.csv']:
    try:
        df=pd.read_csv(ROOT/name); add(f'csv:{name}','PASS' if len(df)>0 else 'WARN',f'rows={len(df)}'); numeric=df.select_dtypes(include=[np.number]); add(f'finite:{name}','PASS' if numeric.empty or np.isfinite(numeric.to_numpy()).all() else 'FAIL','numeric values finite')
    except Exception as exc: add(f'csv:{name}','FAIL',str(exc))
for name in ['road_metrics.json','calibration.json','topology.json','opendrive_validation_report.json','processing_report.json']:
    try: json.loads((ROOT/name).read_text()); add(f'json:{name}','PASS','valid JSON')
    except Exception as exc: add(f'json:{name}','FAIL',str(exc))
cap_out=cv2.VideoCapture(str(ROOT/'annotated_adas_video.mp4')); readable=cap_out.isOpened(); out_frames=int(cap_out.get(cv2.CAP_PROP_FRAME_COUNT)) if readable else 0; ok,_=cap_out.read() if readable else (False,None); cap_out.release(); add('annotated_video_content','PASS' if readable and out_frames>0 and ok else 'FAIL',f'frames={out_frames}')
metric_scale_unknown=scale_info.get('status')=='UNKNOWN'; speed_df=pd.read_csv(ROOT/'speed_profile.csv'); vehicle_df=pd.read_csv(ROOT/'vehicle_tracks.csv'); cal=json.loads((ROOT/'calibration.json').read_text()); road_data=json.loads((ROOT/'road_metrics.json').read_text());
add('assumed_speed_detected','FAIL' if any('30' in str(x).lower() for x in speed_df.to_dict('records')) else 'PASS','no nominal speed literal'); add('assumed_scale_labeled','PASS' if scale_info.get('status') in {'UNKNOWN','ASSUMED','ESTIMATED','MEASURED'} else 'FAIL',str(scale_info)); add('metric_leakage','FAIL' if metric_scale_unknown and (speed_df['speed_mps'].notna().any() or ('distance_m' in vehicle_df and vehicle_df['distance_m'].notna().any())) else 'PASS','metric fields gated by scale'); add('zero_fake_speed','WARN' if speed_df['speed_relative_units_s'].dropna().eq(0).all() and len(speed_df['speed_relative_units_s'].dropna())>1 else 'PASS','relative speed is not fabricated zero-filled'); add('calibration_status_fields','PASS' if all(isinstance(v,dict) and {'value','unit','status','confidence','uncertainty','source'}<=set(v) for v in cal.values()) else 'FAIL','calibration provenance fields'); add('lane_system_label','PASS' if 'multi-lane geometric reconstruction from segmentation/marking evidence + depth + temporal tracking' in json.dumps(MODEL_SELECTION) else 'FAIL','not claiming dedicated learned 3D lane detector')
try:
    parsed=etree.fromstring((ROOT/'road.xodr').read_bytes()); road_nodes=parsed.findall('road'); geom_nodes=parsed.findall('./road/planView/geometry'); xodr_len=float(road_nodes[0].get('length')) if road_nodes else -1; geom_len=sum(float(g.get('length')) for g in geom_nodes); add('xodr_structure','PASS' if len(road_nodes)==1 else 'FAIL',f'roads={len(road_nodes)}'); add('xodr_length_consistency','PASS' if abs(xodr_len-geom_len)<1e-5 else 'FAIL',f'road={xodr_len} geometry_sum={geom_len}'); add('xodr_finite_geometry','PASS' if all(np.isfinite(float(g.get(k))) for g in geom_nodes for k in ('s','x','y','hdg','length')) else 'FAIL','all geometry finite'); xodr_lane_count=len(parsed.findall('./road/lanes/laneSection/right/lane')); csv_lane_count=int(road_data.get('lane_count_best_supported',0)); add('lane_xodr_consistency','WARN' if xodr_lane_count>max(0,csv_lane_count-1) else 'PASS',f'xodr={xodr_lane_count} summary={csv_lane_count}')
except Exception as exc: add('xodr_parse','FAIL',str(exc))
vo_df=pd.read_csv(ROOT/'ego_trajectory.csv'); add('implausible_acceleration','WARN' if 'acceleration_quality' in vo_df and (vo_df['acceleration_quality']=='IMPLAUSIBLE').any() else 'PASS','physical plausibility flag'); displacement=vo_df['frame_displacement_relative'].dropna(); add('vo_drift_quality','WARN' if len(displacement)>3 and displacement.median()>0 and displacement.max()>20*displacement.median() else 'PASS','relative step outlier check')
critical=['artifact:annotated_adas_video.mp4','artifact:road.xodr','annotated_video_content','xodr_structure','xodr_length_consistency']; critical_ok=all(checks[k]['status']=='PASS' for k in critical if k in checks); has_fail=any(v['status']=='FAIL' for v in checks.values()); overall='FAIL' if has_fail else ('PASS' if critical_ok else 'PARTIAL'); checks['overall']={'status':overall,'detail':'production-oriented monocular ADAS reconstruction prototype content and semantic gate'}
processing={'model_selection':MODEL_SELECTION,'report':report,'validation':checks,'artifact_dir':str(PROJECT_ROOT),'archive':str(ARCHIVE_PATH),'prototype_status':'production-oriented monocular ADAS reconstruction prototype'}; (ROOT/'processing_report.json').write_text(json.dumps(processing,indent=2,default=str)); shutil.make_archive(str(ARCHIVE_PATH.with_suffix('')),'zip',root_dir=PROJECT_ROOT.parent,base_dir=PROJECT_ROOT.name); add('complete_results.zip','PASS' if ARCHIVE_PATH.exists() and ARCHIVE_PATH.stat().st_size>0 else 'FAIL',f'bytes={ARCHIVE_PATH.stat().st_size if ARCHIVE_PATH.exists() else 0}'); processing['validation']=checks; (ROOT/'processing_report.json').write_text(json.dumps(processing,indent=2,default=str)); print(json.dumps({'overall':overall,'project_root':str(PROJECT_ROOT),'zip':str(ARCHIVE_PATH),'validation':checks},indent=2,default=str))
from IPython.display import display,FileLink
display(FileLink(str(ARCHIVE_PATH)))